In [4]:
# Tell Jupyter to automatically reload external modules before running the cell
%load_ext autoreload
%autoreload 2

# The proper, bulletproof way to import from a subfolder in Python
from scripts import telemetry_utils as tu

# Print a status message
print("1. Loading raw ISS telemetry data...")
print("2. Running regional bounding box tracker...")

# Run the function on our raw data
regional_counts = tu.count_regional_flyovers('./data/iss_data.csv')

print("\nSUCCESS!")
print("Here are the total telemetry pings recorded over our targeted regions:")

# Print out the dictionary cleanly
for region, count in regional_counts.items():
    print(f"- {region}: {count} data points")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
1. Loading raw ISS telemetry data...
2. Running regional bounding box tracker...

SUCCESS!
Here are the total telemetry pings recorded over our targeted regions:
- East Lansing: 0 data points
- Michigan: 9 data points
- Chicago: 0 data points
- Illinois: 7 data points


In [1]:
import pandas as pd

print("--- Sandbox: Sovereign Airspace Analysis ---")
df = pd.read_csv('./data/iss_data.csv')

ocean_count = 0
land_count = 0

# I want to explicitly check every region to see if the ISS is over water or land.
# It's a bit of a broad net to just look for 'Ocean' or 'Sea', but it's a solid starting point.
for index, row in df.iterrows():
    region_name = str(row['region'])
    
    if ("Ocean" in region_name or "Sea" in region_name):
        ocean_count = ocean_count + 1
    else:
        land_count = land_count + 1

total_points = len(df)
ocean_percent = (ocean_count / total_points) * 100.0
land_percent = (land_count / total_points) * 100.0

print(f"Time spent over international waters: {ocean_percent:.2f}%")
print(f"Time spent over sovereign land: {land_percent:.2f}%")

--- Sandbox: Sovereign Airspace Analysis ---
Time spent over international waters: 69.87%
Time spent over sovereign land: 30.13%


In [2]:
import pandas as pd

print("\n--- Sandbox: Orbital Physics Correlation ---")
df = pd.read_csv('./data/iss_data.csv')

# I need to clean the data before running any math on it.
# Forcing everything to be a float so the correlation calculation doesn't break.
df['altitude_km'] = pd.to_numeric(df['altitude_km'], errors='coerce')
df['speed_kmph'] = pd.to_numeric(df['speed_kmph'], errors='coerce')

# Let's drop the NaNs so our calculations are clean.
clean_df = df.dropna(subset=['altitude_km', 'speed_kmph'])

# Using pandas built-in correlation tool here instead of writing it from scratch
# because the math behind Pearson correlation is pretty standard and robust.
correlation = clean_df['altitude_km'].corr(clean_df['speed_kmph'])

print(f"Pearson Correlation between Altitude and Speed: {correlation:.4f}")
if (correlation < 0):
    print("This negative correlation makes sense! As altitude drops due to drag, velocity increases.")


--- Sandbox: Orbital Physics Correlation ---
Pearson Correlation between Altitude and Speed: -0.9130
This negative correlation makes sense! As altitude drops due to drag, velocity increases.


In [3]:
import pandas as pd

print("\n--- Sandbox: Day/Night Cycle Approximation ---")
df = pd.read_csv('./data/iss_data.csv')

# Making sure the timestamp is an actual datetime object so I can extract the exact hour.
df['timestamp'] = pd.to_datetime(df['timestamp'])

day_count = 0
night_count = 0

# I'm going to iterate through and approximate local time based on longitude.
# 15 degrees of longitude equals exactly 1 hour of time shift.
for index, row in df.iterrows():
    utc_time = row['timestamp']
    lon = row['longitude']
    
    hour_offset = lon / 15.0
    utc_hour_decimal = utc_time.hour + (utc_time.minute / 60.0)
    local_solar_hour = (utc_hour_decimal + hour_offset) % 24.0
    
    # Assuming rough daylight hours are between 6 AM and 6 PM local solar time.
    if (local_solar_hour >= 6.0 and local_solar_hour <= 18.0):
        day_count = day_count + 1
    else:
        night_count = night_count + 1

print(f"Total telemetry pings in Daylight: {day_count}")
print(f"Total telemetry pings in Eclipse (Night): {night_count}")


--- Sandbox: Day/Night Cycle Approximation ---
Total telemetry pings in Daylight: 4581
Total telemetry pings in Eclipse (Night): 4059
